# Overview

This figure shows the performance of the best performing
elevation-\>$D/K$ model and the distribution of $D/K$ Values

## Data source

``` example
analysis/all_test_performance.csv
```

For the plot

``` example
analysis/overall_performance.csv
```

For selecting run

# Setup

``` python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from neural_spd.config import PROJECT_ROOT
from neural_spd import plot_styles
plot_styles.apply()
# todo directory stuff
PERF_METRICS_PATH = PROJECT_ROOT / "analysis/overall_performance.csv"
RAW_PERF_PATH = PROJECT_ROOT / "analysis/all_test_performance.csv"
```

# Load and select data

``` python
perf_metrics_df = pd.read_csv(PERF_METRICS_PATH)
# select row in data frame where target=DoK, data=elevation, noise=0, and nrmse is lowest
best_perf_row = perf_metrics_df[
    (perf_metrics_df["target"] == "DoK") &
    (perf_metrics_df["data"] == "elevation") &
    (perf_metrics_df["noise"] == 0)
].sort_values("nrmse").iloc[0]
# select seed from row
best_seed = best_perf_row["seed"]
raw_perf_df = pd.read_csv(RAW_PERF_PATH)
# select rows where seed is best_seed target is DoK data is elevation and noise is 0
best_raw_perf_df = raw_perf_df[
    (raw_perf_df["seed"] == best_seed) &
    (raw_perf_df["target"] == "DoK") &
    (raw_perf_df["data"] == "elevation") &
    (raw_perf_df["noise"] == 0)
]
```

# Plotting function

``` python
def plot_raw_perf(ax):
    #loglogspace
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("True $(D/K)$")
    ax.set_ylabel("Inferred $(D/K)$")
    sns.kdeplot(
        data=best_raw_perf_df,
        x="true_labels",
        y="predictions",
        ax=ax,
        fill=True,
        alpha=1,
        levels=5,
        label="Performance of Elevation->$(D/K)$ Network",
    )
    ax2 = ax.twinx()
    ax2.set_ylabel("")
    ax2.spines["right"].set_visible(True)
    sns.histplot(
        data=best_raw_perf_df,
        x="true_labels",
        color=plot_styles.COLORS['blue'],
        ax=ax2,
        alpha=0.25,
        edgecolor="none",
        label="True $(D/K)$ Distribution",
    )
    # one-to-one line
    ax.plot(
        [best_raw_perf_df["true_labels"].min(), best_raw_perf_df["true_labels"].max()],
        [best_raw_perf_df["true_labels"].min(), best_raw_perf_df["true_labels"].max()],
        color=plot_styles.COLORS["gray_mid"],
        linestyle="--",
        alpha=0.5,
        #line size
        linewidth=0.5,
        label="One-to-One Line"
    )
```

# Generate Plots

``` python
fig, ax = plt.subplots(figsize=(10, 6))
plot_raw_perf(ax)
#fig.tight_layout()
fig.legend()
fig.show()
```